In [1]:
# core/engine/lib/llm/agents/_02_router/test.ipynb

# Setup project path and log in as admin.
from neurocad.utils.testing import req
req.login_test()

=== ОТЛАДКА TESTING.PY ===
Текущая рабочая директория: /root/projects/aleksmir-dev/neurocad/neurocad/core/engine/lib/llm/agents/_02_router
Путь к test/.env, который мы ищем: /root/projects/aleksmir-dev/neurocad/test/.env
APP_PORT из os.getenv: 8000
[TEST] Logging in as test...
[TEST] Login successful


True

In [10]:
import asyncio
from neurocad.core.engine.lib.llm.agents._02_router.service import CoreEngineLibLlmAgentsRouterService

print("Доступные агенты в сервисе роутера:")
for name, desc in CoreEngineLibLlmAgentsRouterService.AGENTS.items():
    print(f" - {name}: {desc}")

Доступные агенты в сервисе роутера:
 - none: Запрос не связан с созданием личной страницы: анекдоты, погода, политика, математика, посторонние разговоры. Сюда же попадают вопросы про работу самого редактора (как сохранить, как поменять цвет).
 - personal: Создать персональную страницу (сайт-визитку, портфолио). Запросы вроде: 'сделай мой личный сайт', 'хочу страницу с моими проектами', 'сайт-визитка для меня'.


In [2]:
# Test router ASK mode (new request).
payload = {
    "user_message": "Хочу сделать личный сайт-портфолио",
    "history": []
}
result = req.post("/core/engine/lib/llm/agents/_02_router/route", payload)

[TEST] POST http://127.0.0.1:8012/core/engine/lib/llm/agents/_02_router/route
[TEST] Status: 200
{
  "success": true,
  "data": {
    "agent": null,
    "question": "Отлично! Уточните, пожалуйста — вам нужна именно персональная страница-портфолио с вашими проектами и информацией о вас?"
  }
}


In [3]:
# Test router PICK mode (reply to a clarifying question).
payload_pick = {
    "user_message": "Да, мне нужна именно персональная страница с моими проектами",
    "history": [
        {"role": "user", "content": "Хочу сделать личный сайт-портфолио"},
        {"role": "assistant", "content": result["data"]["question"], "is_quest": 1}
    ]
}
result_pick = req.post("/core/engine/lib/llm/agents/_02_router/route", payload_pick)

[TEST] POST http://127.0.0.1:8012/core/engine/lib/llm/agents/_02_router/route
[TEST] Status: 200
{
  "success": true,
  "data": {
    "agent": "personal",
    "question": null
  }
}


In [4]:
# Test router off-topic (should choose 'none').
payload = {
    "user_message": "Расскажи мне анекдот про программистов",
    "history": []
}
result = req.post("/core/engine/lib/llm/agents/_02_router/route", payload)

[TEST] POST http://127.0.0.1:8012/core/engine/lib/llm/agents/_02_router/route
[TEST] Status: 200
{
  "success": true,
  "data": {
    "agent": null,
    "question": "Уточните, пожалуйста — вам просто нужен анекдот для развлечения, или вы хотите создать личную страницу (например, портфолио)?"
  }
}


In [6]:
# Test router PICK mode (off-topic reply).
payload_offtopic = {
    "user_message": "Анекдот",
    "history": [
        {"role": "user", "content": "Расскажи мне анекдот про программистов"},
        {"role": "assistant", "content": result["data"]["question"], "is_quest": 1}
    ]
}
result_offtopic = req.post("/core/engine/lib/llm/agents/_02_router/route", payload_offtopic)

[TEST] POST http://127.0.0.1:8012/core/engine/lib/llm/agents/_02_router/route
[TEST] Status: 200
{
  "success": true,
  "data": {
    "agent": "none",
    "question": null
  }
}
